# Embedding Module Test
This notebook verifies that our `SentenceTransformerEmbedder` correctly loads the BGE-M3 model, utilizes the Apple Silicon (MPS) GPU, and successfully processes text chunks into 1024-dimensional vectors.

In [1]:
import sys
import os
from pathlib import Path

# Ensure the src directory is in the Python path so we can import our package
src_path = str(Path(os.getcwd()).parent / 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from arxiv_scholar.embedding.st_embedder import SentenceTransformerEmbedder

/Users/tri/Projects/arxiv-scholar/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialize the embedder
# It will automatically resolve to 'mps' (Apple GPU) if running locally
embedder = SentenceTransformerEmbedder(model_name="BAAI/bge-m3", device="auto")

print("✅ Embedder Initialized")
print(f"Model: {embedder.model_name}")
print(f"Compute Device: {embedder.device}")
print(f"Vector Dimension: {embedder.dimension}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34827.09it/s]


✅ Embedder Initialized
Model: BAAI/bge-m3
Compute Device: mps
Vector Dimension: 1024


/Users/tri/Projects/arxiv-scholar/src/arxiv_scholar/embedding/st_embedder.py:101: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self._model.get_sentence_embedding_dimension()


In [3]:
# Create some sample chunks simulating scientific paper text
sample_chunks = [
    "We introduce a new sliding window attention mechanism for long-context language models.",
    "The retrieval performance on the MTEB benchmark was significantly improved compared to BM25.",
    "The loss is calculated as $\\mathcal{L} = \\sum_{i=1}^{n} \\nabla_\\theta \\log P(y_i|x_i)$ over the batch."
]

print(f"Embedding {len(sample_chunks)} sample chunks...")
vectors = embedder.embed(sample_chunks)

print(f"\n✅ Embedding Complete")
print(f"Total vectors generated: {len(vectors)}")
print(f"Length of first vector: {len(vectors[0])}")
print(f"First 5 raw float values of vector 1: {vectors[0][:5]}")

Embedding 3 sample chunks...

✅ Embedding Complete
Total vectors generated: 3
Length of first vector: 1024
First 5 raw float values of vector 1: [-0.07382694631814957, -0.02848309837281704, -0.03295493498444557, 0.01592371240258217, -0.015077188611030579]


In [4]:
# Test semantic similarity between two highly related texts
import numpy as np
from numpy.linalg import norm

related_texts = [
    "Machine learning models are heavily reliant on large datasets for training.",
    "Deep learning algorithms require massive amounts of data to learn effectively."
]

print("Embedding related texts...")
related_vectors = embedder.embed(related_texts)

# Calculate cosine similarity
vec1, vec2 = related_vectors[0], related_vectors[1]
cosine_sim = np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))

print(f"Cosine Similarity: {cosine_sim:.4f}")
assert cosine_sim > 0.7, "Similarity should be high for related texts!"
print("✅ Semantic similarity test passed!")

Embedding related texts...
Cosine Similarity: 0.7510
✅ Semantic similarity test passed!
